# Assignment 4 (new data) — 0. Inventory of `data/`

**Before any model is trained, establish what is actually on disk.**

The datasets in `data/` arrive with their own `*_info.md` description files. Those files describe what the
datasets *are*, not necessarily what was *downloaded*. This notebook trusts neither: it walks the
directories, counts files, opens images and profiles the CSV, and reports the result.

That distinction is not pedantry. A partial download, a renamed folder or a missing split all look fine
until a training run fails on them — and it is much cheaper to find that out here than three hours in.

| # | Dataset | Structure | Model family | Source |
|---|---------|-----------|--------------|--------|
| **A** | CDC BRFSS diabetes 2015+2023 | tabular, 546,166 × 19 | **MLP** | `data/` |
| **B** | Rice Image Dataset | image, 3×32×32, 5 classes | **CNN** | `data/` |
| **C** | MNIST | image, 1×28×28, 10 classes | **CNN** | `torchvision`, auto-download |

Two of the three live in `data/` and have to be copied onto the machine by hand. MNIST does not —
`ass4_utils.load_mnist` fetches it on first use — so the only thing to check for it is that the download
works and the arrays arrive in the expected shape.

In [ ]:
import os, sys, json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))        # notebook/  -> ass4_newdata
sys.path.insert(0, os.path.abspath(".."))       # repo root  -> ass4_utils, scratch_nn

import ass4_newdata as D
import ass4_utils as U

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 130)

print("repo root:", D.ROOT)
print("data dir :", D.DATA)

---
## 1. What is on disk

`D.inventory()` walks every declared directory and counts what it really contains. The
`usable_for_training` column is the one that matters: an image split is only usable if it has **class
folders**, because the folder name *is* the label.

In [ ]:
inv = D.inventory()
display(inv)

print("\ntotal items on disk:", f"{inv['items'].sum():,}")
print("usable for supervised training:",
      f"{inv.loc[inv['usable_for_training'], 'items'].sum():,}")

missing = inv[~inv["present"]]
if len(missing):
    print("\nMISSING - copy data/ across before running notebooks 01-02:")
    display(missing)
else:
    print("\nevery declared part is present")

---
## 2. Dataset A — BRFSS diabetes (tabular)

The claims in `data/diabetes_info.md` that the notebooks depend on are checked here directly: the row
count, the absence of missing values, the class split, and the 2015/2023 balance.

In [ ]:
df = pd.read_csv(D.DIABETES_CSV)

print(f"shape            {df.shape}")
print(f"missing values   {int(df.isna().sum().sum())}")
print(f"duplicate rows   {int(df.duplicated().sum()):,}  (kept on purpose - see info file)")
print(f"memory           {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

display(df.head())
display(df.describe().T[["min", "max", "mean", "std"]].round(2))

In [ ]:
# Target balance overall and per survey year - the imbalance drives how the
# results in notebook 01 have to be read.
counts = df[D.BRFSS_TARGET].value_counts().sort_index()
by_year = pd.crosstab(df["year"], df[D.BRFSS_TARGET], normalize="index")
by_year.columns = D.BRFSS_CLASSES

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
bars = ax[0].bar(D.BRFSS_CLASSES, counts.values, color=["#4C78A8", "#F58518", "#E45756"])
ax[0].set_title("Target balance (Diabetes_012)")
ax[0].set_ylabel("respondents")
for b, c in zip(bars, counts.values):
    ax[0].text(b.get_x() + b.get_width() / 2, c, f"{c:,}\n{c / counts.sum():.1%}",
               ha="center", va="bottom", fontsize=8)

by_year.plot(kind="bar", stacked=True, ax=ax[1],
             color=["#4C78A8", "#F58518", "#E45756"])
ax[1].set_title("Class share by survey year")
ax[1].set_ylabel("share"); ax[1].tick_params(axis="x", rotation=0)
ax[1].legend(fontsize=7)
fig.tight_layout(); plt.show()

display((by_year * 100).round(2))
print("\nRows per year:"); print(df["year"].value_counts().sort_index().to_string())

The two survey years agree closely on every prevalence, which is the evidence that merging them vertically
was legitimate. The class split is roughly **84 / 2 / 14**, and `prediabetes` is rare enough that accuracy
alone will be a misleading score — the point `theory_notes.md` §4.2 makes.

---
## 3. Dataset B — Rice images

75,000 JPEGs in five class folders. What matters for the loader is that every image really is the same
size and colour mode, because the CNN legs assume a fixed tensor shape.

In [ ]:
rice_counts = {c: D._count_images(os.path.join(D.RICE_DIR, c)) for c in D.RICE_CLASSES}
print("images per class:")
for c, n in rice_counts.items():
    print(f"  {c:<12}{n:>8,}")
print(f"  {'TOTAL':<12}{sum(rice_counts.values()):>8,}")
print(f"\nperfectly balanced: {len(set(rice_counts.values())) == 1}")

# Verify the declared 250x250 RGB on a random sample rather than assuming it.
from PIL import Image
rng = np.random.default_rng(U.SEED)
sizes, modes = {}, {}
for c in D.RICE_CLASSES:
    folder = os.path.join(D.RICE_DIR, c)
    files = sorted(os.listdir(folder))
    for f in rng.choice(files, size=60, replace=False):
        with Image.open(os.path.join(folder, f)) as im:
            sizes[im.size] = sizes.get(im.size, 0) + 1
            modes[im.mode] = modes.get(im.mode, 0) + 1
print("\nsampled 300 images ->")
print("  sizes:", sizes)
print("  modes:", modes)

In [ ]:
# One example per class at native resolution, then the same grain at the 32x32
# the CNN notebooks actually train on.
fig, axes = plt.subplots(2, len(D.RICE_CLASSES), figsize=(13, 5.4))
for j, c in enumerate(D.RICE_CLASSES):
    folder = os.path.join(D.RICE_DIR, c)
    f = sorted(os.listdir(folder))[0]
    with Image.open(os.path.join(folder, f)) as im:
        im = im.convert("RGB")
        axes[0, j].imshow(im)
        axes[1, j].imshow(im.resize((32, 32), Image.BILINEAR))
    axes[0, j].set_title(c, fontsize=10)
    for i in (0, 1):
        axes[i, j].axis("off")
axes[0, 0].set_ylabel("250x250"); axes[1, 0].set_ylabel("32x32")
fig.suptitle("Rice grains: native resolution (top) and the 32x32 training input (bottom)")
fig.tight_layout(); plt.show()

**Why 32×32 is enough here.** What separates the five varieties is grain *shape*, *size* and *colour* —
Basmati is long and slender, Arborio short and round. Those survive aggressive downsampling, as the bottom
row shows. Each image is one centred grain on a black background, so there is no clutter to resolve and no
segmentation step needed. 32×32 also makes the full 75,000-image set fit comfortably in memory as float32
(~0.9 GB) and keeps the NumPy-from-scratch leg tractable, which a 250×250 input would not.

---
## 4. Dataset C — MNIST

MNIST is not in `data/` and does not need to be: `ass4_utils.load_mnist` fetches it through `torchvision`
on first call (about 12 MB) into `data/mnist/`. The only check worth making is that the download works and
the arrays arrive in the shape the notebooks expect.

It is the right third dataset for two reasons. It differs from the rice images in every way that matters
for the comparison — **greyscale rather than RGB, ten classes rather than five, handwriting rather than
photography** — so the collection covers two genuinely different image problems. And the repository root
already ran MNIST with the same architecture in `02_mnist.ipynb`, which lets notebook 03 check its own
numbers against a committed result.

In [ ]:
try:
    Xm, ym, Xmt, ymt, mmeta = U.load_mnist(n_train=2_000, n_test=500)
    print(f"shape {mmeta['shape']}  classes {len(mmeta['classes'])}  dtype {Xm.dtype}")
    print(f"per-class counts (2k sample): {np.bincount(ym)}")
    U.plot_samples(Xm, ym, mmeta["classes"], n=12, title="MNIST - the third dataset")
    plt.show()
except Exception as e:
    print("MNIST download failed:", type(e).__name__, e)
    print("torchvision fetches it on first use; check the network connection.")

---
## 5. Verdict

In [ ]:
verdict = pd.DataFrame([
    ["A  BRFSS diabetes", "tabular", "546,166 rows x 19 features", "3", "data/",
     "01_diabetes_brfss.ipynb"],
    ["B  Rice images",    "image",   "75,000 x 250x250 RGB",       "5", "data/",
     "02_rice_cnn.ipynb"],
    ["C  MNIST",          "image",   "70,000 x 28x28 greyscale",   "10", "torchvision",
     "03_mnist_cnn.ipynb"],
], columns=["dataset", "kind", "material", "classes", "source", "notebook"])
display(verdict)

**All three datasets are usable, and they divide the assignment's argument cleanly.**

- **BRFSS diabetes** is the control case: a tabular problem with no spatial neighbourhood, so convolution
  has nothing to exploit and notebook 01 uses a plain MLP. It is also the badly imbalanced one, which is
  what makes macro F1 rather than accuracy the honest headline there.
- **Rice** and **MNIST** are the two image problems, and they are deliberately unalike — colour versus
  greyscale, five classes versus ten, pre-segmented grains versus handwriting. Both are close to balanced,
  so accuracy means what it appears to mean.

Notebook 04 then runs the M1→M4 improvement ladder from `theory_notes.md` §3, and notebook 05 collects
everything into the cross-dataset comparison.